# Chapter 2 Lab: Rule 90 — Local Rules to Global Patterns

## 1. Goal

Observe how local rule, iteration, interaction, and state jointly generate a global pattern. Predict every perturbation before running it.

## 2. Theory

An elementary cellular automaton stores binary state in a one-dimensional row. Each cell reads a three-cell neighborhood and updates simultaneously. Rule 90 is equivalent to XOR of the left and right neighbors.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def elementary_step(state: np.ndarray, rule_number: int) -> np.ndarray:
    next_state = np.zeros_like(state)
    table = np.array([(rule_number >> pattern) & 1 for pattern in range(8)], dtype=np.uint8)
    neighborhoods = (state[:-2] << 2) | (state[1:-1] << 1) | state[2:]
    next_state[1:-1] = table[neighborhoods]
    return next_state

def simulate(initial_state: np.ndarray, rule_number: int, steps: int) -> np.ndarray:
    history = np.zeros((steps, len(initial_state)), dtype=np.uint8)
    state = initial_state.copy()
    for t in range(steps):
        history[t] = state
        state = elementary_step(state, rule_number)
    return history

def show(history: np.ndarray, title: str) -> None:
    plt.figure(figsize=(10, 6))
    plt.imshow(history, cmap='binary', interpolation='nearest', aspect='auto')
    plt.title(title)
    plt.xlabel('position')
    plt.ylabel('iteration')
    plt.show()

## 3. Baseline: One Seed, Rule 90

Before running: sketch the first five rows. Predict whether the pattern remains symmetric.

In [ ]:
width = 101
steps = 50
one_seed = np.zeros(width, dtype=np.uint8)
one_seed[width // 2] = 1

rule90_history = simulate(one_seed, rule_number=90, steps=steps)
show(rule90_history, 'Rule 90 — one central seed')

## 4. Perturb the State

Activate two adjacent cells. Predict which symmetry changes while the local rule stays fixed.

In [ ]:
two_seeds = np.zeros(width, dtype=np.uint8)
two_seeds[width // 2] = 1
two_seeds[width // 2 + 1] = 1

two_seed_history = simulate(two_seeds, rule_number=90, steps=steps)
show(two_seed_history, 'Rule 90 — two adjacent seeds')

## 5. Perturb the Local Rule

Change Rule 90 to Rule 91, which flips one entry in the eight-pattern lookup table. Predict whether the global difference remains local.

In [ ]:
rule91_history = simulate(one_seed, rule_number=91, steps=steps)
show(rule91_history, 'Rule 91 — one rule-table bit changed')

## 6. Perturb the Update Process

Synchronous updates read the old row everywhere. The function below writes left-to-right, so later cells can read a value already changed in the same iteration. Predict the effect before running.

In [ ]:
def asynchronous_rule90_step(state: np.ndarray) -> np.ndarray:
    next_state = state.copy()
    for i in range(1, len(state) - 1):
        next_state[i] = next_state[i - 1] ^ state[i + 1]
    next_state[0] = 0
    next_state[-1] = 0
    return next_state

def simulate_asynchronous(initial_state: np.ndarray, steps: int) -> np.ndarray:
    history = np.zeros((steps, len(initial_state)), dtype=np.uint8)
    state = initial_state.copy()
    for t in range(steps):
        history[t] = state
        state = asynchronous_rule90_step(state)
    return history

async_history = simulate_asynchronous(one_seed, steps=steps)
show(async_history, 'Rule 90 logic — asynchronous left-to-right update')

## 7. Reflection

For each experiment, identify which element changed: Local Rule, Iteration, Interaction, State, or update process.

1. Which prediction was least accurate, and why?
2. Did a one-bit rule change remain a one-cell visual change?
3. Why does update order count as part of the system definition?
4. What does the automaton omit when used as an analogy for neural networks or markets?

## 8. Extensions

- Compare Rules 30, 90, and 110.
- Track the fraction of active cells over time. Does one number preserve the pattern?
- Design a metric that makes a smooth change look abrupt, connecting the experiment to the Research Corner.